# A7: Full-Text Search

In [5]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     /home/rodrigofrancachaves/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/rodrigofrancachaves/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/rodrigofrancachaves/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/rodrigofrancachaves/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/rodrigofrancachaves/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Part 1: The Offline Component

Parse the XML file. Store the results into a list of dictionaries called all_docs.

In [6]:
from xml.etree import ElementTree as ET

def preprocess_text(text):
    """Apply stopword filtering and lemmatization to text."""
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    
    # Tokenize
    tokens = word_tokenize(text.lower())
    
    # Remove stopwords and lemmatize; keep only alphabetic tokens
    processed = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token.isalpha() and token not in stop_words
    ]
    
    return ' '.join(processed)


def parse_pubmed_xml(file_path):
    """
    Parse a PubMed XML file and return a list of dictionaries.
    Each dict has keys: 'pmid', 'title', 'abstract'.
    The abstract is preprocessed (stopword removal + lemmatization).
    """
    tree = ET.parse(file_path)
    root = tree.getroot()
    
    all_docs = []
    
    for article in root.findall('.//PubmedArticle'):
        # Extract PMID
        pmid_elem = article.find('.//PMID')
        pmid = pmid_elem.text.strip() if pmid_elem is not None else None
        
        # Extract Title
        title_elem = article.find('.//ArticleTitle')
        title = ''.join(title_elem.itertext()).strip() if title_elem is not None else ''
        
        # Extract Abstract — may have multiple AbstractText elements
        abstract_elems = article.findall('.//AbstractText')
        abstract_raw = ' '.join(
            ''.join(elem.itertext()).strip()
            for elem in abstract_elems
            if elem is not None
        ).strip()
        
        # Apply preprocessing to abstract
        abstract_processed = preprocess_text(abstract_raw) if abstract_raw else ''
        
        if pmid is not None:
            all_docs.append({
                'pmid': pmid,
                'title': title,
                'abstract': abstract_processed
            })
    
    return all_docs


# Parse the XML file
XML_FILE = 'pubmed25n0001 (2).xml'
all_docs = parse_pubmed_xml(XML_FILE)
print(f'Parsed {len(all_docs)} documents.')

Parsed 30000 documents.


Index the information in all_docs to an ElasticSearch index.

In [9]:
from elasticsearch import Elasticsearch, helpers

# Connect to local ElasticSearch instance
es = Elasticsearch('http://localhost:9200')

INDEX_NAME = 'pubmed'

# Define the index mapping
index_mapping = {
    'mappings': {
        'properties': {
            'pmid':     {'type': 'keyword'},
            'title':    {'type': 'text'},
            'abstract': {'type': 'text'}
        }
    }
}

# Delete existing index if present, then recreate
try:
    es.indices.delete(index=INDEX_NAME)
    print(f'Deleted existing index: {INDEX_NAME}')
except Exception:
    print(f'Index {INDEX_NAME} did not exist, creating fresh.')

es.indices.create(index=INDEX_NAME, mappings=index_mapping['mappings'])
print(f'Created index: {INDEX_NAME}')

# Bulk index all documents
actions = [
    {
        '_index': INDEX_NAME,
        '_id': doc['pmid'],
        '_source': doc
    }
    for doc in all_docs
]

success, failed = helpers.bulk(es, actions)
print(f'Indexed {success} documents successfully. Failed: {failed}')

es.indices.refresh(index=INDEX_NAME)

Deleted existing index: pubmed
Created index: pubmed
Indexed 30000 documents successfully. Failed: []


ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

## Part 1 Questions

1) How many documents did you index in total?
2) For document with PMID=22, find its title and its abstract, afte applying stopwords filtering and lemmatization.

In [10]:
# Question 1: Total number of indexed documents
count_result = es.count(index=INDEX_NAME)
total_docs = count_result['count']
print(f'Question 1: Total documents indexed = {total_docs}')

# Question 2: Document with PMID=22
result = es.get(index=INDEX_NAME, id='22')
doc_22 = result['_source']

print('\nQuestion 2: Document with PMID=22')
print(f'Title    : {doc_22["title"]}')
print(f'Abstract : {doc_22["abstract"]}')

Question 1: Total documents indexed = 30000

Question 2: Document with PMID=22
Title    : [Demonstration of tumor inhibiting properties of a strongly immunostimulating low-molecular weight substance. Comparative studies with ifosfamide on the immuno-labile DS carcinosarcoma. Stimulation of the autoimmune activity for approx. 20 days by BA 1, a N-(2-cyanoethylene)-urea. Novel prophylactic possibilities].
Abstract : report given recent discovery outstanding immunological property ba low molecular mass experiment d carcinosarcoma bearing wistar rat shown ba dosage percent mg kg negligible lethality percent result recovery rate percent without hyperglycemia one test percent hyperglycemia otherwise unchanged condition reference substance ifosfamide development cyclophosphamide applied without hyperglycemia efficient dosage percent mg kg brought recovery rate percent lethality percent contrary ba hyperglycemia caused improvement recovery rate however comparison characterized fact substance e